In [1]:
from google.colab import files
files.upload()

Saving TrainingNames.txt to TrainingNames.txt


{'TrainingNames.txt': b'Aarav\r\nVihaan\r\nVivaan\r\nAnanya\r\nDiya\r\nAdvik\r\nSaanvi\r\nIshaan\r\nMyra\r\nKabir\r\nKiara\r\nAryan\r\nAnika\r\nReyansh\r\nNavya\r\nKrishna\r\nAmaira\r\nArjun\r\nShanaya\r\nAtharv\r\nKyra\r\nShaurya\r\nIra\r\nDhruv\r\nPrisha\r\nArush\r\nRiya\r\nVeer\r\nAavya\r\nIshani\r\nAyaan\r\nZara\r\nRudra\r\nPari\r\nKian\r\nSara\r\nYuvan\r\nAvni\r\nVivaan\r\nSiya\r\nDarsh\r\nMyra\r\nIshit\r\nAnvi\r\nVedant\r\nAahil\r\nAmara\r\nAdvait\r\nAisha\r\nShivansh\r\nInaya\r\nRanveer\r\nZoya\r\nSai\r\nMeher\r\nAyaan\r\nSana\r\nHridaan\r\nAdvika\r\nKavya\r\nTanish\r\nJiya\r\nYatharth\r\nMishka\r\nRidhaan\r\nVanya\r\nMoksh\r\nAanya\r\nAaryan\r\nPihu\r\nAarush\r\nKhushi\r\nShlok\r\nTanvi\r\nLaksh\r\nShanvi\r\nVihaan\r\nBhavya\r\nArnav\r\nRiva\r\nAgastya\r\nAira\r\nDaksh\r\nAhana\r\nAyush\r\nZiva\r\nSiddharth\r\nNyra\r\nNihal\r\nIshita\r\nAdvik\r\nTrisha\r\nShiv\r\nRhea\r\nVed\r\nSaisha\r\nIshan\r\nGauri\r\nMadhav\r\nAdvika\r\nVivaan\r\nPrisha\r\nVeer\r\nMyra\r\nArjun\r\nSiya\r\n

In [2]:
!pip install torch

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [3]:
def load_names(file_path):
    with open(file_path, "r") as f:
        return [line.strip().lower() for line in f if line.strip()]

names = load_names("TrainingNames.txt")
print("Total names:", len(names))

Total names: 4415


In [4]:
def build_vocab(names):
    chars = sorted(list(set("".join(names))))
    chars = ['<s>', '</s>'] + chars

    char2idx = {c: i for i, c in enumerate(chars)}
    idx2char = {i: c for c, i in char2idx.items()}

    return char2idx, idx2char

char2idx, idx2char = build_vocab(names)
vocab_size = len(char2idx)

def encode(name):
    return [char2idx['<s>']] + [char2idx[c] for c in name] + [char2idx['</s>']]

dataset = [encode(n) for n in names]

In [5]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.emb(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden


class CharBiLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        x = self.emb(x)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out


class CharRNNWithAttention(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.emb(x)
        out, _ = self.rnn(x)

        attn_weights = torch.softmax(self.attn(out), dim=1)
        context = attn_weights * out

        out = self.fc(context)
        return out

In [6]:
def train_model(model, dataset, epochs=8, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for seq in dataset:
            x = torch.tensor(seq[:-1]).unsqueeze(0)
            y = torch.tensor(seq[1:]).unsqueeze(0)

            optimizer.zero_grad()

            if isinstance(model, CharRNN):
                output, _ = model(x)
            else:
                output = model(x)

            loss = criterion(output.view(-1, output.size(-1)), y.view(-1))

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.2f}")

In [7]:
def generate_name(model, max_len=15, temperature=0.8):

    model.eval()

    idx = char2idx['<s>']
    input = torch.tensor([[idx]])

    name = ""
    hidden = None

    for _ in range(max_len):

        if isinstance(model, CharRNN):
            output, hidden = model(input, hidden)
        else:
            output = model(input)

        logits = output[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        # prevent early stopping
        if len(name) < 3:
            probs[char2idx['</s>']] = 0

        probs = probs / probs.sum()

        idx = torch.multinomial(probs, 1).item()
        char = idx2char[idx]

        # skip special tokens
        if char in ['<s>', '</s>']:
            if char == '</s>' and len(name) >= 3:
                break
            else:
                continue

        name += char
        input = torch.tensor([[idx]])

    return name

In [8]:
def novelty_rate(generated, train_set):
    if len(generated) == 0:
        return 0.0
    return len([n for n in generated if n not in train_set]) / len(generated)


def diversity(names):
    if len(names) == 0:
        return 0.0
    return len(set(names)) / len(names)

In [11]:
hidden_size = 64

models = {
    "RNN": CharRNN(vocab_size, hidden_size),
    "BiLSTM": CharBiLSTM(vocab_size, hidden_size),
    "Attention": CharRNNWithAttention(vocab_size, hidden_size)
}

#  MODEL-SPECIFIC SETTINGS
config = {
    "RNN": {"epochs": 12, "temp": 0.6},
    "BiLSTM": {"epochs": 8, "temp": 1.2},
    "Attention": {"epochs": 10, "temp": 1.1}
}

results = {}

for model_name, model in models.items():
    print(f"\n===== Training {model_name} =====")

    epochs = config[model_name]["epochs"]
    temp = config[model_name]["temp"]

    train_model(model, dataset, epochs=epochs)

    generated = []
    attempts = 0
    max_attempts = 300

    while len(generated) < 100 and attempts < max_attempts:
        name_gen = generate_name(model, temperature=temp)

        if len(name_gen) >= 2:
            generated.append(name_gen)

        attempts += 1

    if len(generated) == 0:
        print(" Model failed")
        generated = ["unknown"] * 10

    novelty = novelty_rate(generated, set(names))
    div = diversity(generated)

    results[model_name] = (novelty, div)

    print("\nSample names:")
    print(generated[:10])

    print(f"Novelty: {novelty:.4f}")
    print(f"Diversity: {div:.4f}")


===== Training RNN =====
Epoch 1/12, Loss: 7610.99
Epoch 2/12, Loss: 7057.66
Epoch 3/12, Loss: 6801.29
Epoch 4/12, Loss: 6604.19
Epoch 5/12, Loss: 6435.50
Epoch 6/12, Loss: 6425.98
Epoch 7/12, Loss: 6293.90
Epoch 8/12, Loss: 6224.59
Epoch 9/12, Loss: 6232.98
Epoch 10/12, Loss: 6179.71
Epoch 11/12, Loss: 6152.70
Epoch 12/12, Loss: 6092.42

Sample names:
['shubhang', 'shushma', 'shushma', 'shulan', 'shuchita', 'shwetani', 'shushma', 'shuhara', 'shubha', 'shruta']
Novelty: 0.1200
Diversity: 0.4200

===== Training BiLSTM =====
Epoch 1/8, Loss: 1088.05
Epoch 2/8, Loss: 31.29
Epoch 3/8, Loss: 10.75
Epoch 4/8, Loss: 7.02
Epoch 5/8, Loss: 4.64
Epoch 6/8, Loss: 1.81
Epoch 7/8, Loss: 1.27
Epoch 8/8, Loss: 2.44

Sample names:
['opi', 'olp', 'oll', 'lll', 'ezz', 'ijj', 'oli', 'fel', 'ooq', 'osa']
Novelty: 0.9800
Diversity: 0.6800

===== Training Attention =====
Epoch 1/10, Loss: 6534.92
Epoch 2/10, Loss: 3347.70
Epoch 3/10, Loss: 1935.18
Epoch 4/10, Loss: 1251.12
Epoch 5/10, Loss: 882.77
Epoch 6/

In [ ]:
print("\n FINAL RESULTS\n")

print("{:<10} {:<10} {:<10}".format("Model", "Novelty", "Diversity"))
print("-" * 30)

for m, (n, d) in results.items():
    print(f"{m:<10} {n:.4f}     {d:.4f}")

In [13]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, model in models.items():
    print(name, count_params(model))

RNN 11932
BiLSTM 171292
Attention 16092
